## NOTEBOOK 1: FEATURE EXTRACTION & PREPROCESSING

### Pipeline Structure
1. Load PCAP or CSV from OpenPLC Lab
2. Parse Modbus/TCP (port 502) and S7Comm (port 102) protocols
3. Extract features from packet payloads
4. Handle class imbalance with SMOTE
5. Normalize and reshape data for ML vs DL models
6. Output: processed_dataset.npz with both 2D (ML) and 3D (DL) formats

In [45]:
import joblib
import json
import os
import numpy as np
import pandas as pd
import warnings
from scapy.all import rdpcap, TCP, IP
from scapy.utils import PcapReader 
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import Pipeline
from sklearn.neighbors import NearestNeighbors

warnings.filterwarnings('ignore')
SEED = 42
np.random.seed(SEED)

#### SECTION 1: PROTOCOL PARSERS

In [46]:
class ModbusParser:
    """
    Parser for Modbus/TCP protocol (port 502).
    Extracts MBAP header and PDU fields.
    """
    MODBUS_PORT = 502
    
    @staticmethod
    def extract_features(payload, pkt_length, timestamp):
        """
        Extract Modbus/TCP features from payload.
        
        MBAP Header (7 bytes):
            [0:2] Transaction ID
            [2:4] Protocol ID (always 0x0000)
            [4:6] Length
            [6:7] Unit ID
            [7:8] Function Code
        
        Returns: dict with extracted features or None if invalid
        """
        if payload is None or len(payload) < 8:
            return None
        
        try:
            trans_id = int.from_bytes(payload[0:2], byteorder='big')
            proto_id = int.from_bytes(payload[2:4], byteorder='big')
            length = int.from_bytes(payload[4:6], byteorder='big')
            unit_id = payload[6]
            func_code = payload[7]
            
            # Validate Modbus protocol ID
            if proto_id != 0:
                return None
            
            # Extract starting address and quantity based on function code
            ref_num = 0
            word_cnt = 1
            
            if len(payload) >= 10:
                ref_num = int.from_bytes(payload[8:10], byteorder='big')
            if len(payload) >= 12:
                word_cnt = int.from_bytes(payload[10:12], byteorder='big')
            
            # Calculate payload size (excluding Modbus header)
            payload_size = max(0, len(payload) - 8)
            
            return {
                'packet_length': pkt_length,
                'timestamp': timestamp,
                'transaction_id': trans_id,
                'protocol_id': proto_id,
                'unit_id': unit_id,
                'function_code': func_code,
                'reference_num': ref_num,
                'word_count': word_cnt,
                'payload_size': payload_size,
                'protocol': 'modbus'
            }
        except (ValueError, IndexError):
            return None


class S7CommParser:
    """
    Parser for Siemens S7Communication protocol (port 102).
    Accounts for TPKT (4 bytes) and COTP headers.
    """
    S7COMM_PORT = 102
    S7_MAGIC = 0x32  # S7Comm magic byte
    
    @staticmethod
    def extract_features(payload, pkt_length, timestamp):
        if payload is None or len(payload) < 14:
            return None
        
        try:
            # TPKT Header is 4 bytes [0:4]. COTP length byte is at payload[4].
            cotp_len = payload[4]
            s7_offset = 4 + 1 + cotp_len  # TPKT (4) + Length byte (1) + COTP body
            
            if len(payload) < s7_offset + 10:
                return None
            
            proto_id = payload[s7_offset]
            if proto_id != S7CommParser.S7_MAGIC:
                return None
            
            pdu_type = payload[s7_offset + 1]
            pdu_length = int.from_bytes(payload[s7_offset + 6:s7_offset + 8], byteorder='big')
            pdu_ref = int.from_bytes(payload[s7_offset + 4:s7_offset + 6], byteorder='big')
            
            function_code = payload[s7_offset + 8] if len(payload) > s7_offset + 8 else 0
            subfunction = payload[s7_offset + 9] if len(payload) > s7_offset + 9 else 0
            
            area_code = 0
            if len(payload) > s7_offset + 12:
                area_code = payload[s7_offset + 12]
            
            payload_size = max(0, len(payload) - s7_offset)
            
            return {
                'packet_length': pkt_length,
                'timestamp': timestamp,
                'protocol_id': proto_id,
                'pdu_type': pdu_type,
                'pdu_reference': pdu_ref,
                'function_code': function_code,
                'subfunction': subfunction,
                'area_code': area_code,
                'pdu_length': pdu_length,
                'payload_size': payload_size,
                'protocol': 's7comm'
            }
        except (ValueError, IndexError):
            return None


class S7CommPlusParser:
    """
    Parser for Siemens S7Communication Plus protocol (port 102).
    Accounts for TPKT and COTP encap offsets.
    """
    S7COMMPLUS_PORT = 102
    S7PLUS_MAGIC = 0x72  # S7Comm+ magic byte
    
    @staticmethod
    def extract_features(payload, pkt_length, timestamp):
        if payload is None or len(payload) < 16:
            return None
        
        try:
            # Check for TPKT header
            if payload[0] != 0x03:
                return None
            
            # COTP is fixed at 3 bytes: [0x02][0xF0][0x80]
            # Or it could be variable length [0x02][0xF0][0x80][0x00] etc.
            # The S7+ magic byte is at offset 7 (TPKT=4 + COTP=3)
            
            # Method 1: Try fixed offset 7
            s7_offset = 7
            
            # Check if magic byte is at offset 7
            if len(payload) > s7_offset and payload[s7_offset] == S7CommPlusParser.S7PLUS_MAGIC:
                proto_id = payload[s7_offset]
            else:
                # Method 2: Search for 0x72 in the first 30 bytes
                s7_offset = -1
                for i in range(4, min(len(payload), 30)):
                    if payload[i] == S7CommPlusParser.S7PLUS_MAGIC:
                        # Verify it's a valid S7+ header
                        if i + 5 < len(payload):
                            # Check for valid version (0x01) and PDU type (0x02)
                            version = payload[i + 1] if i + 1 < len(payload) else 0
                            pdu_type = payload[i + 3] if i + 3 < len(payload) else 0
                            if version in [0x00, 0x01] and pdu_type in [0x01, 0x02]:
                                s7_offset = i
                                break
                
                if s7_offset == -1:
                    return None
                
                proto_id = payload[s7_offset]
            
            if proto_id != S7CommPlusParser.S7PLUS_MAGIC:
                return None
            
            # Now parse the S7+ message starting from s7_offset
            # S7+ Header: [0x72][version][reserved][pdu_type][reserved][reserved][seq_hi][seq_lo][opcode][reserved][func_hi][func_lo]
            if len(payload) < s7_offset + 12:
                return None
            
            proto_version = payload[s7_offset + 1] if s7_offset + 1 < len(payload) else 0
            pdu_type = payload[s7_offset + 3] if s7_offset + 3 < len(payload) else 0
            
            # Sequence number (2 bytes)
            seq_hi = payload[s7_offset + 6] if s7_offset + 6 < len(payload) else 0
            seq_lo = payload[s7_offset + 7] if s7_offset + 7 < len(payload) else 0
            sequence = (seq_hi << 8) | seq_lo
            
            # Opcode (0x31=Request, 0x32=Response, 0x33=Notification)
            opcode = payload[s7_offset + 8] if s7_offset + 8 < len(payload) else 0
            
            # Function code (2 bytes)
            func_hi = payload[s7_offset + 10] if s7_offset + 10 < len(payload) else 0
            func_lo = payload[s7_offset + 11] if s7_offset + 11 < len(payload) else 0
            function_code = (func_hi << 8) | func_lo
            
            # Extract message ID from data block (first byte after header)
            data_start = s7_offset + 12
            message_id = payload[data_start] if len(payload) > data_start else 0
            
            # Calculate payload size
            payload_size = max(0, len(payload) - s7_offset)
            
            # Try to extract more features if available
            var_spec = 0
            var_length = 0
            is_request = (function_code >> 15) & 1
            
            # Look for data length in trailer
            pdu_length = 0
            if len(payload) > s7_offset + 16:
                # Trailer starts at the end: [0x72][pdu_type][length_hi][length_lo]
                trailer_start = len(payload) - 4
                if trailer_start > s7_offset:
                    trailer_magic = payload[trailer_start] if trailer_start < len(payload) else 0
                    if trailer_magic == 0x72:
                        pdu_length_hi = payload[trailer_start + 2] if trailer_start + 2 < len(payload) else 0
                        pdu_length_lo = payload[trailer_start + 3] if trailer_start + 3 < len(payload) else 0
                        pdu_length = (pdu_length_hi << 8) | pdu_length_lo
            
            return {
                'packet_length': pkt_length,
                'timestamp': timestamp,
                'protocol_id': proto_id,
                'protocol_version': proto_version,
                'pdu_type': pdu_type,
                'sequence': sequence,
                'opcode': opcode,
                'function_code': function_code,
                'message_id': message_id,
                'is_request': is_request,
                'pdu_length': pdu_length,
                'payload_size': payload_size,
                'protocol': 's7comm_plus'
            }
        except (ValueError, IndexError) as e:
            return None

#### SECTION 2: PCAP EXTRACTION ENGINE

In [47]:
def pcap_to_dataframe(pcap_path, label=0, protocol_type='auto'):
    """
    Extract features from PCAP file using streaming PcapReader.
    
    Args:
        pcap_path: Path to PCAP file
        label: Binary label (0=normal, 1=attack)
        protocol_type: 'modbus', 's7comm', 's7comm_plus', or 'auto' (detect automatically)
    
    Returns: pandas.DataFrame with extracted features
    """
    data = []
    protocol_stats = {'modbus': 0, 's7comm': 0, 's7comm_plus': 0, 'unknown': 0}
    
    try:
        with PcapReader(pcap_path) as pcap_reader:
            for pkt in pcap_reader:
                if not (IP in pkt and TCP in pkt):
                    continue
                
                # IP Layer Extraction
                src_ip = pkt[IP].src
                dst_ip = pkt[IP].dst
                ttl = pkt[IP].ttl
                ip_flags = int(pkt[IP].flags)
                frag_offset = pkt[IP].frag
                
                # TCP Layer Extraction
                sport = pkt[TCP].sport
                dport = pkt[TCP].dport
                tcp_flags = int(pkt[TCP].flags)
                seq_num = pkt[TCP].seq
                ack_num = pkt[TCP].ack
                window_size = pkt[TCP].window
                
                payload = bytes(pkt[TCP].payload) if pkt[TCP].payload else None
                
                # Base network features to attach to every valid packet
                net_features = {
                    'src_ip': src_ip,
                    'dst_ip': dst_ip,
                    'sport': sport,
                    'dport': dport,
                    'ttl': ttl,
                    'ip_flags': ip_flags,
                    'frag_offset': frag_offset,
                    'tcp_flags': tcp_flags,
                    'seq': seq_num,
                    'ack': ack_num,
                    'window_size': window_size,
                    'label': label
                }
                
                # Check both source and destination ports for bi-directional captures
                if dport == ModbusParser.MODBUS_PORT or sport == ModbusParser.MODBUS_PORT:
                    features = ModbusParser.extract_features(payload, len(pkt), pkt.time)
                    if features:
                        features.update(net_features)
                        data.append(features)
                        protocol_stats['modbus'] += 1
                    else:
                        protocol_stats['unknown'] += 1
                
                elif (dport == S7CommParser.S7COMM_PORT or sport == S7CommParser.S7COMM_PORT):
                        if not payload:
                            protocol_stats['unknown'] += 1
                            continue

                        features = None
                        
                        # Try S7comm+ (magic byte 0x72 at offset 7 typically)
                        if protocol_type in ['s7comm_plus', 'auto']:
                            features = S7CommPlusParser.extract_features(payload, len(pkt), pkt.time)
                            if features:
                                protocol_stats['s7comm_plus'] += 1

                        # Try S7comm if not found
                        if not features and protocol_type in ['s7comm', 'auto']:
                            features = S7CommParser.extract_features(payload, len(pkt), pkt.time)
                            if features:
                                protocol_stats['s7comm'] += 1

                        if features:
                            features.update(net_features)
                            data.append(features)
                        else:
                            protocol_stats['unknown'] += 1

    except Exception as e:
        print(f"Error reading PCAP {pcap_path}: {e}")
        return pd.DataFrame()
    
    if not data:
        print(f"Warning: No industrial protocol packets extracted from {pcap_path}")
        print(f"Protocol breakdown: {protocol_stats}")
        return pd.DataFrame()
    
    df = pd.DataFrame(data)
    print(f"Extracted {len(df)} packets from {pcap_path}")
    print(f"Protocol breakdown: {protocol_stats}")
    print(f"Class distribution: {df['label'].value_counts().to_dict()}")
    
    # Calculate time deltas within each protocol group
    if 'timestamp' in df.columns and 'protocol' in df.columns:
        for protocol in df['protocol'].unique():
            mask = df['protocol'] == protocol
            df.loc[mask, 'time_delta'] = df.loc[mask, 'timestamp'].diff().fillna(0)
    
    return df


def load_and_clean_csv(filepath, label_column='label'):
    """
    Args :
        filepath: Path to CSV file
        label_column: Name of column containing labels
    
    Returns: pandas.DataFrame with cleaned data
    """

    print(f"Loading CSV: {filepath}")
    try:
        df = pd.read_csv(filepath)
    except Exception as e:
        print(f"Error reading CSV {filepath}: {e}")
        return pd.DataFrame()
    
    print(f"Loaded shape: {df.shape}")
    print(f"Columns: {list(df.columns)}")
    
    # Handle missing values
    initial_rows = len(df)
    df = df.dropna()
    dropped_rows = initial_rows - len(df)
    if dropped_rows > 0:
        print(f"Dropped {dropped_rows} rows with missing values")
    
    # Remove duplicates
    df = df.drop_duplicates()
    
    # Check class distribution
    if label_column in df.columns:
        print(f"Class distribution:\n{df[label_column].value_counts()}")
    
    return df

#### SECTION 3: FEATURE NORMALIZATION & SEQUENCE CREATION

In [ ]:
def align_features(df, target_protocol='modbus'):
    """
    Align features for both Modbus and S7Comm protocols.
    Fills missing protocol-specific columns with 0.
    
    Args:
        df: DataFrame with mixed protocols
        target_protocol: 'modbus', 's7comm', or 'all'
    
    Returns: Aligned DataFrame with consistent columns
    """
    # Shared numerical network-layer features across all protocols
    net_cols = [
        'sport', 'dport', 'ttl', 'ip_flags', 
        'frag_offset', 'tcp_flags', 'seq', 'ack', 'window_size'
    ]

    # Standard columns for ML models
    if target_protocol == 'modbus':
        feature_cols = [
            'packet_length', 'time_delta', 'transaction_id', 'unit_id',
            'function_code', 'reference_num', 'word_count', 'payload_size'
        ] + net_cols
    elif target_protocol == 's7comm':
        feature_cols = [
            'packet_length', 'time_delta', 'pdu_type', 'pdu_reference',
            'function_code', 'subfunction', 'area_code', 'pdu_length', 'payload_size'
        ] + net_cols
    else:  # 'all' - unified feature set
        feature_cols = [
            'packet_length', 'time_delta', 'function_code', 'payload_size'
        ] + net_cols
    
    # Ensure all feature columns exist
    for col in feature_cols:
        if col not in df.columns:
            df[col] = 0
    
    return df[feature_cols + ['label']]


# SECTION 3: FEATURE NORMALIZATION & SEQUENCE CREATION
# ====================================================================
def create_sequences(X, y, time_steps=10):
    """
    Transform 2D matrix [samples, features] to 3D [samples, time_steps, features].
    
    IMPORTANT: This creates sequences by taking consecutive samples.
    For the sequences to be meaningful, the data MUST be in chronological order
    AND grouped by network flow/protocol.
    
    If your data is shuffled, this will create RANDOM sequences!
    """
    Xs, ys = [], []
    for i in range(len(X) - time_steps):
        Xs.append(X[i:(i + time_steps)])
        # Use the last element's label as the sequence label
        ys.append(y[i + time_steps])
    return np.array(Xs), np.array(ys)

def create_flow_based_sequences(df, time_steps=10):
    """
    Create sequences based on network flows.
    Each sequence comes from the same source/destination pair.
    
    Args:
        df: DataFrame with packet data (MUST have src_ip, dst_ip, sport, dport)
        time_steps: Number of packets per sequence
    
    Returns:
        X_seq, y_seq: Sequences and their labels
    """
    import pandas as pd
    import numpy as np
    
    # Create flow identifier (5-tuple if available, else use simpler grouping)
    if all(col in df.columns for col in ['src_ip', 'dst_ip', 'sport', 'dport', 'protocol']):
        # 5-tuple flow
        df['flow_id'] = df['src_ip'] + '_' + df['dst_ip'] + '_' + df['sport'].astype(str) + '_' + df['dport'].astype(str) + '_' + df['protocol'].astype(str)
    elif all(col in df.columns for col in ['src_ip', 'dst_ip']):
        # Simpler flow (IP pair only)
        df['flow_id'] = df['src_ip'] + '_' + df['dst_ip']
    else:
        # Fallback: use all samples as one group (not ideal)
        print("[!] Warning: No flow identifiers found. Creating sequences from entire dataset.")
        df['flow_id'] = 'all_flows'
    
    # Sort by timestamp within each flow
    if 'timestamp' in df.columns:
        df = df.sort_values(['flow_id', 'timestamp'])
    
    all_sequences = []
    all_labels = []
    
    for flow_id, group in df.groupby('flow_id'):
        # Get features and labels for this flow
        # Exclude non-feature columns
        exclude_cols = ['flow_id', 'src_ip', 'dst_ip', 'sport', 'dport', 'protocol', 
                       'timestamp', 'label', 'source_protocol']
        feature_cols_flow = [col for col in group.columns if col not in exclude_cols]
        X_flow = group[feature_cols_flow].values
        y_flow = group['label'].values
        
        # Create sequences from this flow
        if len(X_flow) >= time_steps:
            for i in range(len(X_flow) - time_steps):
                # Check if all labels in sequence are the same (optional)
                # This prevents mixing attack types in one sequence
                if len(np.unique(y_flow[i:i+time_steps])) == 1:
                    all_sequences.append(X_flow[i:i+time_steps])
                    all_labels.append(y_flow[i + time_steps - 1])  # Last packet's label
    
    if len(all_sequences) == 0:
        print(f"[!] No sequences created. Minimum length per flow: {time_steps}")
        print("Falling back to basic sequence creation...")
        return create_sequences(df.drop(columns=['flow_id'], errors='ignore').values, 
                               df['label'].values, time_steps)
    
    X_seq = np.array(all_sequences)
    y_seq = np.array(all_labels)
    
    print(f"[+] Created {len(X_seq)} flow-based sequences")
    print(f"   Sequence shape: {X_seq.shape}")
    
    return X_seq, y_seq


# HELPER FUNCTIONS FOR RARE CLASS AUGMENTATION
# ====================================================================

def detect_feature_indices(feature_cols):
    """
    Detect important feature indices for augmentation
    """
    indices = {}
    
    # Common feature names to look for
    feature_mapping = {
        'packet_length': ['packet_length', 'Length', 'length'],
        'sport': ['sport', 'src_port', 'source_port'],
        'dport': ['dport', 'dst_port', 'dest_port'],
        'tcp_flags': ['tcp_flags', 'flags'],
        'ttl': ['ttl', 'time_to_live'],
        'seq': ['seq', 'sequence'],
        'ack': ['ack', 'acknowledgement'],
        'window_size': ['window_size', 'window']
    }
    
    # Find indices
    for feature, possible_names in feature_mapping.items():
        for name in possible_names:
            if name in feature_cols:
                indices[feature] = feature_cols.index(name)
                break
    
    return indices

def get_class_name(class_id):
    """
    Get class name from class ID
    """
    ATTACK_TYPES = {
        0: "Normal / Benign",
        1: "MITM / PLC Attack", 
        2: "Port Scan / Reconnaissance",
        3: "Telnet PLC Attack",
        4: "Web Access Attack",
        5: "DDoS / DoS",
        6: "Replay Attack"
    }
    return ATTACK_TYPES.get(class_id, f"Class_{class_id}")

#### SECTION 4: MAIN PIPELINE EXECUTION

Output : processed_dataset.npz

It Contains preprocessed 2D arrays (X_train_ML, X_test_ML, y_train_ML, y_test_ML) for machine learning and 3D sequences (X_train_DL, X_test_DL, y_train_DL, y_test_DL) for deep learning models. 

In [ ]:
if __name__ == "__main__":
    SEED = 42  # Explicitly defined random state seed
    
    print("=" * 70)
    print("PREPROCESSING PIPELINE: FEATURE EXTRACTION & PREPARATION")
    print("=" * 70)
    
    # STEP 1: LOAD DATASETS 
    print("\n[STEP 1/5] --->----->------> Loading Datasets")
    
    datasets = []
    protocol_summary = {}
    
    # PCAP files loaded directly from Datasets_wireshark/
    pcap_files_with_protocol = [
        ('../Datasets_wireshark/modbus_cap.pcapng', 'modbus', 'Real Modbus/TCP traffic', 0),
        ('../Datasets_wireshark/s7comm_cap.pcapng', 's7comm', 'Real S7Comm traffic (snap7)', 0),
        ('../Datasets_wireshark/s7plus_cap.pcapng', 's7plus_semi_authentic', 'Semi-Authentic S7Comm+ (TLV + session)', 0),
        ('../Datasets_wireshark/nmap_cap.pcap', 'nmap_pcap', 'nmap ip/port scan: stealth, agressive,...  ', 2),
        ('../Datasets_wireshark/bettercap_cap.pcap', 'bettercap_pcap', 'MITM flase data injection - Replay attack', 1),
        ('../Datasets_wireshark/metasploit_cap.pcap', 'metasploit_pcap', 'Metasploit attack exploits scenarios', 1)
    ]
    
    # Load normal/attack traffic from PCAP captures
    print("\n[Loading PCAP Captures from Datasets_wireshark/]")
    for pcap_path, protocol_name, description, default_label in pcap_files_with_protocol:
        try:
            df_pcap = pcap_to_dataframe(pcap_path, label=default_label)
            if not df_pcap.empty:
                df_pcap['source_protocol'] = protocol_name
                datasets.append(df_pcap)
                protocol_summary[protocol_name] = {
                    'samples': len(df_pcap),
                    'description': description
                }
                print(f"✓ Loaded: {pcap_path} ({len(df_pcap)} samples)")
                print(f"  Protocol: {description}")
        except FileNotFoundError:
            print(f"⚠ File not found (optional): {pcap_path}")
        except Exception as e:
            print(f"⚠ Error loading {pcap_path}: {e}")
    
    # Load attack traffic from CSV files
    print("\n[Loading Attack Traffic from CSV - HISTORICAL ICS SCENARIOS]")
    attack_csv_files = {
        'MITM_PLCHMI': '../Datasets/Real-Time ICS Cyber with Hacking Scenarios/MITM PLCHMI Attack.csv',
        'TELNET_PLC': '../Datasets/Real-Time ICS Cyber with Hacking Scenarios/TELNET PLC Attack.csv',
        'WEB_Access': '../Datasets/Real-Time ICS Cyber with Hacking Scenarios/WEB Access PLC Attack.csv',
        'Dataset': '../Datasets/dataset.csv/Dataset.csv'
    }

    LABEL_MAPPING = {
        'normal': 0, 'benign': 0, 0: 0,
        'mitm': 1, 'mitm_plchmi': 1,
        'ip_scan': 2, 'port_scan': 2, 'nmap': 2, 'reconnaissance': 2,
        'telnet_plc': 3, 'telnet': 3,
        'web_access': 4, 'http': 4,
        'ddos': 5, 'dos': 5,
        'replay': 6, 'bettercap': 6, 'metasploit': 6
    }
    
    for attack_name, csv_path in attack_csv_files.items():
        try:
            df_attack = load_and_clean_csv(csv_path, label_column='Label')
            if not df_attack.empty:
                existing_label_col = next((c for c in df_attack.columns if c.lower() in ['label', 'target', 'class']), None)
                
                if existing_label_col:
                    if existing_label_col != 'label':
                        df_attack.rename(columns={existing_label_col: 'label'}, inplace=True)
                    df_attack['label'] = df_attack['label'].astype(str).str.lower().map(LABEL_MAPPING).fillna(1)
                else:
                    df_attack['label'] = LABEL_MAPPING.get(attack_name.lower(), 1)
                
                if 'protocol' not in df_attack.columns:
                    df_attack['protocol'] = 'mixed'
                    
                df_attack['source_protocol'] = f'attack_{attack_name}'
                datasets.append(df_attack)
                
                pos_count = (df_attack['label'] == 1).sum()
                neg_count = (df_attack['label'] == 0).sum()
                
                protocol_summary[f'attack_{attack_name}'] = {
                    'samples': len(df_attack),
                    'description': f'Historical attack: {attack_name} (Attacks: {pos_count}, Normal: {neg_count})'
                }
                print(f"✓ Loaded: {attack_name} ({len(df_attack)} samples | Attacks: {pos_count}, Normal: {neg_count})")
                
        except FileNotFoundError:
            print(f"⚠ File not found (optional): {csv_path}")
        except Exception as e:
            print(f"⚠ Error loading CSV {csv_path}: {e}")

    # Summary of loaded data sources
    print("\n[DATA SOURCES LOADED]")
    for source, info in protocol_summary.items():
        print(f"  • {source}: {info['samples']} samples - {info['description']}")
    
    # Fallback / Execution block
    if not datasets:
        print("\n[WARNING] No datasets found! Please check file paths in Datasets_wireshark/ or Datasets/.")
    else:
        # Concatenate all datasets
        print(f"\n[Data Summary] Loaded {len(datasets)} dataset source(s)")
        df_combined = pd.concat(datasets, ignore_index=True)
        
        excluded_cols = [
            'label', 'protocol', 'timestamp', 'source_protocol', 
            'src_ip', 'dst_ip', 'ip.src', 'ip.dst', 'Source', 'Destination',
            # Add these to prevent data leakage:
            'IT_B_Label', 'IT_M_Label', 'NST_B_Label', 'NST_M_Label',
            'sAddress', 'rAddress', 'sMACs', 'rMACs', 'sIPs', 'rIPs',
            'No.', 'Time', 'Info','source_protocol',  
            'frag_offset', 'is_request', 'sUrgRate', 'rUrgRate',
            'sFragmentRate', 'rFragmentRate'
        ]      
        
        # Extract features and filter out any string/object columns (IPs, Strings)
        candidate_features = [col for col in df_combined.columns if col not in excluded_cols]
        numeric_df = df_combined[candidate_features].select_dtypes(include=[np.number])
        feature_cols = numeric_df.columns.tolist()
        
        dropped_string_cols = list(set(candidate_features) - set(feature_cols))
        if dropped_string_cols:
            print(f"\n[INFO] Dropped non-numeric/IP string columns: {dropped_string_cols}")

        X_raw = numeric_df.fillna(0).values
        y_raw = df_combined['label'].values.astype(int)
        
        print(f"Combined dataset shape: {X_raw.shape}")
        print(f"Feature count ({len(feature_cols)}): {feature_cols}")
        print(f"Label distribution: Normal={np.sum(y_raw==0)}, Attack={np.sum(y_raw!=0)}")
        
        if 'source_protocol' in df_combined.columns:
            print(f"\nProtocol diversity in combined dataset:")
            for proto, count in df_combined['source_protocol'].value_counts().items():
                pct = 100 * count / len(df_combined)
                print(f"  • {proto}: {count} samples ({pct:.1f}%)")
    
        # STEP 2: TRAIN/TEST SPLIT
        print("\n[STEP 2/5] --->----->------> Train/Test Split & Stratification")
        
        X_train_raw, X_test_raw, y_train_raw, y_test = train_test_split(
            X_raw, y_raw, test_size=0.2, random_state=SEED, stratify=y_raw
        )
        
        print(f"Train set: {X_train_raw.shape[0]} samples")
        print(f"Test set: {X_test_raw.shape[0]} samples")
        print(f"Train class distribution: {np.bincount(y_train_raw)}")
        print(f"Test class distribution: {np.bincount(y_test)}")
        
        # STEP 3: NORMALIZATION
        print("\n[STEP 3/5] --->----->------> Feature Normalization")
        
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train_raw)
        X_test_scaled = scaler.transform(X_test_raw)
        
        print(f"Scaling applied successfully (StandardScaler)")
        print(f"Train mean: {X_train_scaled.mean(axis=0)[:3]}... (first 3 features)")
        print(f"Train std: {X_train_scaled.std(axis=0)[:3]}... (first 3 features)")
        
        # STEP 4: CLASS IMBALANCE HANDLING WITH SMART AUGMENTATION
        print("\n[STEP 4/5] --->----->------> Handling Class Imbalance (SMOTE)")
        
        # Check current class distribution
        class_counts = np.bincount(y_train_raw)
        print(f"\nOriginal class distribution:")
        ATTACK_TYPES = {
            0: "Normal / Benign",
            1: "MITM / PLC Attack",
            2: "Port Scan / Reconnaissance",
            3: "Telnet PLC Attack",
            4: "Web Access Attack",
            5: "DDoS / DoS",
            6: "Replay Attack"
        }
        for class_id, count in enumerate(class_counts):
            class_name = ATTACK_TYPES.get(class_id, f"Class_{class_id}")
            print(f"  Class {class_id} ({class_name}): {count} samples ({100*count/len(y_train_raw):.1f}%)")
        
        # Identify rare classes (less than 50 samples)
        RARE_CLASS_THRESHOLD = 50
        rare_classes = [i for i, count in enumerate(class_counts) if count < RARE_CLASS_THRESHOLD]
        
        if rare_classes:
            print(f"\nRare classes detected: {rare_classes}")
            print(f"These classes will be augmented with synthetic variations.")
            
            # Generate synthetic samples for rare classes
            def generate_nmap_synthetic_samples(X_template, class_id, n_samples=500):
                """Generate realistic nmap-like samples with controlled variations."""
                if len(X_template) == 0:
                    return np.array([]), np.array([])
                
                if len(X_template) > 1:
                    template = X_template.mean(axis=0)
                    std = X_template.std(axis=0)
                else:
                    template = X_template[0]
                    std = np.ones_like(template) * 0.1
                
                synthetic_samples = []
                port_scan_ports = [21, 22, 23, 25, 80, 443, 8080, 3306, 5432, 3389, 5900, 6667]
                tcp_flags = [2, 18, 4, 16, 20]
                packet_sizes = [40, 44, 48, 52, 56, 60, 64, 68, 72, 76]
                
                for i in range(n_samples):
                    sample = template.copy()
                    
                    # Add realistic variations for nmap scans
                    if len(sample) >= 1:
                        sample[0] = np.random.choice(packet_sizes) + np.random.uniform(-5, 5)
                    
                    if len(sample) >= 2:
                        sample[1] = np.random.randint(1024, 65535)
                    
                    if len(sample) >= 3:
                        sample[2] = np.random.choice(port_scan_ports + list(range(10000, 20000)))
                    
                    if len(sample) >= 4:
                        sample[3] = np.random.choice(tcp_flags)
                    
                    # Add small random noise
                    noise_scale = 0.05 * std + 0.01
                    sample += np.random.normal(0, noise_scale, len(sample))
                    
                    synthetic_samples.append(sample)
                
                return np.array(synthetic_samples), np.full(n_samples, class_id)
            
            def generate_general_synthetic_samples(X_template, class_id, n_samples=500):
                """Generate synthetic samples for other rare classes."""
                if len(X_template) == 0:
                    return np.array([]), np.array([])
                
                if len(X_template) > 1:
                    template = X_template.mean(axis=0)
                    std = X_template.std(axis=0)
                    std = np.maximum(std, 0.01)
                else:
                    template = X_template[0]
                    std = np.ones_like(template) * 0.1
                
                synthetic_samples = []
                for i in range(n_samples):
                    variation = np.random.normal(0, 0.1 * std, len(template))
                    sample = template + variation
                    synthetic_samples.append(sample)
                
                return np.array(synthetic_samples), np.full(n_samples, class_id)
            
            print("\n[Augmenting Rare Classes]")
            
            # Separate data by class
            X_by_class = {}
            for class_id in np.unique(y_train_raw):
                X_by_class[class_id] = X_train_scaled[y_train_raw == class_id]
            
            augmented_X = []
            augmented_y = []
            
            for class_id in rare_classes:
                X_class = X_by_class.get(class_id, np.array([]))
                class_name = ATTACK_TYPES.get(class_id, f"Class_{class_id}")
                
                print(f"\n  Augmenting Class {class_id} ({class_name}):")
                print(f"     Original samples: {len(X_class)}")
                
                if len(X_class) == 0:
                    print(f"     No samples found, skipping")
                    continue
                
                target_samples = 1000
                
                # Determine augmentation strategy
                if class_id == 2 or "scan" in class_name.lower() or "nmap" in class_name.lower():
                    X_aug, y_aug = generate_nmap_synthetic_samples(X_class, class_id, target_samples)
                    print(f"     Using nmap-specific augmentation")
                else:
                    X_aug, y_aug = generate_general_synthetic_samples(X_class, class_id, target_samples)
                    print(f"     Using general augmentation")
                
                if len(X_aug) > 0:
                    augmented_X.append(X_aug)
                    augmented_y.append(y_aug)
                    print(f"     Generated {len(X_aug)} synthetic samples")
            
            # Combine augmented data with original data
            if augmented_X:
                X_train_scaled = np.vstack([X_train_scaled] + augmented_X)
                y_train_raw = np.hstack([y_train_raw] + augmented_y)
                print(f"\nTotal augmentation added: {sum(len(x) for x in augmented_X)} samples")
                print(f"New training set size: {len(X_train_scaled)} samples")
            else:
                print("\nNo augmentation applied")
        
        # Apply SMOTE with adaptive parameters
        print("\n[Applying SMOTE with Adaptive Parameters]")
        
        class_counts = np.bincount(y_train_raw)
        min_class_samples = np.min(class_counts[class_counts > 0])
        max_class_samples = np.max(class_counts)
        
        print(f"  Min class samples: {min_class_samples}")
        print(f"  Max class samples: {max_class_samples}")
        
        if max_class_samples / min_class_samples > 1.5:
            k_neighbors = min(3, min_class_samples - 1) if min_class_samples > 1 else 1
            
            target_samples = max_class_samples
            
            sampling_strategy = {}
            for class_id in np.unique(y_train_raw):
                if class_counts[class_id] < target_samples:
                    sampling_strategy[class_id] = target_samples
            
            print(f"  SMOTE target samples per class: {target_samples}")
            print(f"  SMOTE k_neighbors: {k_neighbors}")
            
            try:
                smote = SMOTE(
                    random_state=SEED,
                    k_neighbors=k_neighbors,
                    sampling_strategy=sampling_strategy
                )
                X_train_balanced, y_train_balanced = smote.fit_resample(X_train_scaled, y_train_raw)
                print(f"  SMOTE completed")
            except Exception as e:
                print(f"  SMOTE failed: {e}")
                print("  Using augmentation-only approach")
                X_train_balanced, y_train_balanced = X_train_scaled, y_train_raw
        else:
            print("  Classes are already balanced, skipping SMOTE")
            X_train_balanced, y_train_balanced = X_train_scaled, y_train_raw
        
        # Verify final class distribution
        final_counts = np.bincount(y_train_balanced)
        print(f"\n[Final Class Distribution]")
        for class_id, count in enumerate(final_counts):
            class_name = ATTACK_TYPES.get(class_id, f"Class_{class_id}")
            percentage = 100 * count / len(y_train_balanced)
            print(f"  Class {class_id} ({class_name}): {count} samples ({percentage:.1f}%)")
        
        # Check for remaining issues
        zero_variance_classes = []
        for class_id in np.unique(y_train_balanced):
            class_data = X_train_balanced[y_train_balanced == class_id]
            zero_variance_features = np.sum(np.std(class_data, axis=0) == 0)
            if zero_variance_features > 0:
                zero_variance_classes.append((class_id, zero_variance_features))
                print(f"  Warning: Class {class_id} has {zero_variance_features} features with zero variance")
        
        print(f"\nClass imbalance handling complete")
        print(f"Final training set size: {X_train_balanced.shape[0]} samples")
        print(f"Final feature count: {X_train_balanced.shape[1]} features")
        
        # STEP 5: RESHAPE & SAVE FOR ML vs DL
        print("\n[STEP 5/5] --->----->------> Reshaping and Saving Output")

        # Create 3D sequence shapes explicitly for DL models
        print("\n[Creating Sequences - USING FLOW-BASED METHOD]")

        # First, we need to get the original dataframe with flow info
        try:
            # Try flow-based approach
            X_train_DL, y_train_DL = create_flow_based_sequences(
                df_combined, 
                time_steps=10
            )
            
            # Since we split after feature extraction, we need to track indices
            X_test_DL, y_test_DL = create_sequences(X_test_scaled, y_test, time_steps=10)
            
            print(f"[+] Flow-based sequences created")
            print(f"   Train sequences: {X_train_DL.shape}")
            print(f"   Test sequences: {X_test_DL.shape}")
        except Exception as e:
            print(f"[!] Flow-based creation failed: {e}")
            print("Falling back to basic sequence creation...")
            X_train_DL, y_train_DL = create_sequences(X_train_balanced, y_train_balanced, time_steps=10)
            X_test_DL, y_test_DL = create_sequences(X_test_scaled, y_test, time_steps=10)

        os.makedirs("../outputs", exist_ok=True)
        np.savez_compressed(
            "../outputs/processed_dataset.npz",
            X_train_ML=X_train_balanced, y_train_ML=y_train_balanced,
            X_test_ML=X_test_scaled, y_test_ML=y_test,
            X_train_DL=X_train_DL, y_train_DL=y_train_DL,
            X_test_DL=X_test_DL, y_test_DL=y_test_DL,
            feature_cols=np.array(feature_cols)
        )
        print("✓ Dataset successfully generated and saved to ./outputs/processed_dataset.npz")

PREPROCESSING PIPELINE: FEATURE EXTRACTION & PREPARATION

[STEP 1/5] --->----->------> Loading Datasets

[Loading PCAP Captures from Datasets_wireshark/]
Extracted 2078 packets from ../Datasets_wireshark/modbus_cap.pcapng
Protocol breakdown: {'modbus': 2078, 's7comm': 0, 's7comm_plus': 0, 'unknown': 2085}
Class distribution: {0: 2078}
✓ Loaded: ../Datasets_wireshark/modbus_cap.pcapng (2078 samples)
  Protocol: Real Modbus/TCP traffic


Extracted 2059 packets from ../Datasets_wireshark/s7comm_cap.pcapng
Protocol breakdown: {'modbus': 0, 's7comm': 2059, 's7comm_plus': 0, 'unknown': 2320}
Class distribution: {0: 2059}
✓ Loaded: ../Datasets_wireshark/s7comm_cap.pcapng (2059 samples)
  Protocol: Real S7Comm traffic (snap7)
Extracted 2049 packets from ../Datasets_wireshark/s7plus_cap.pcapng
Protocol breakdown: {'modbus': 0, 's7comm': 0, 's7comm_plus': 2049, 'unknown': 2056}
Class distribution: {0: 2049}
✓ Loaded: ../Datasets_wireshark/s7plus_cap.pcapng (2049 samples)
  Protocol: Semi-Authentic S7Comm+ (TLV + session)
Extracted 12 packets from ../Datasets_wireshark/nmap_cap.pcap
Protocol breakdown: {'modbus': 4, 's7comm': 8, 's7comm_plus': 0, 'unknown': 38}
Class distribution: {2: 12}
✓ Loaded: ../Datasets_wireshark/nmap_cap.pcap (12 samples)
  Protocol: nmap ip/port scan: stealth, agressive,...  
Protocol breakdown: {'modbus': 0, 's7comm': 0, 's7comm_plus': 0, 'unknown': 0}
Extracted 8 packets from ../Datasets_wireshark/me